# NB_05 — Business Insights (SQL + Visualizations)

Reference date: April 1, 2021 (last sale: March 30, 2021)

All queries run against the Gold layer (star schema).  
Charts generated with matplotlib / seaborn on top of pandas DataFrames.

Sections:
1. Sales Year-over-Year
2. Top 10 Products — Revenue & Margin
3. Sales by Division & Region
4. Budget vs Actual by Sales Rep
5. Gross Margin Distribution by Price Tier
6. Top 10 Customers by Revenue
7. Seasonal Sales Pattern (Month × Year heatmap)
8. Data Quality Summary

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Pomerol brand palette ──────────────────────────────────────────────────────
ORANGE  = '#E8531A'
TEAL    = '#2DBD9E'
BLUE    = '#2251CC'
YELLOW  = '#F5A623'
DARK    = '#1A1A1A'
GRAY    = '#6B7280'
CREAM   = '#F5F0EB'
PALETTE = [ORANGE, TEAL, BLUE, YELLOW, '#C0392B', '#27AE60', '#8E44AD', '#2C3E50']

sns.set_theme(style='whitegrid', font='DejaVu Sans')
plt.rcParams.update({
    'figure.facecolor': CREAM,
    'axes.facecolor':   CREAM,
    'axes.edgecolor':   '#D5CFC5',
    'axes.titlesize':   13,
    'axes.titleweight': 'bold',
    'axes.titlecolor':  DARK,
    'axes.labelcolor':  GRAY,
    'axes.labelsize':   10,
    'xtick.color':      GRAY,
    'ytick.color':      GRAY,
    'grid.color':       '#E5E0D8',
    'grid.linewidth':   0.7,
    'font.size':        10,
})

def fmt_millions(x, pos):
    if x >= 1_000_000: return f'${x/1_000_000:.1f}M'
    if x >= 1_000:     return f'${x/1_000:.0f}K'
    return f'${x:.0f}'

def add_pomerol_footer(fig, note=''):
    txt = 'ACME Inc. POC  |  Pomerol  |  April 1, 2021'
    if note: txt = note + '  |  ' + txt
    fig.text(0.5, 0.01, txt, ha='center', fontsize=7, color=GRAY, style='italic')

print('Setup complete. Gold tables ready to query.')

## 1. Sales Year-over-Year

In [ ]:
# ── SQL ──────────────────────────────────────────────────────────────────────
df_yoy = spark.sql("""
    SELECT
        fi.OrderYear,
        fi.FiscalYear,
        ROUND(SUM(fi.SalesAmount), 2)   AS TotalSales,
        ROUND(SUM(fi.GrossMargin), 2)   AS TotalMargin,
        ROUND(
            SUM(fi.GrossMargin) / NULLIF(SUM(fi.SalesAmount), 0) * 100, 1
        )                               AS GrossMarginPct,
        COUNT(DISTINCT fo.OrderID)      AS TotalOrders,
        COUNT(DISTINCT fi.CustomerID)   AS UniqueCustomers
    FROM gold_fact_sales_items fi
    JOIN gold_fact_orders fo ON fi.OrderID = fo.OrderID
    GROUP BY fi.OrderYear, fi.FiscalYear
    ORDER BY fi.OrderYear
""").toPandas()

print(df_yoy.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), facecolor=CREAM)
fig.suptitle('Sales Year-over-Year — ACME Inc.', fontsize=14, fontweight='bold', color=DARK, y=1.01)

years = df_yoy['OrderYear'].astype(str)

# Chart 1 — Revenue vs Margin
ax = axes[0]
w = 0.35
x = np.arange(len(years))
ax.bar(x - w/2, df_yoy['TotalSales'],  w, color=ORANGE, label='Revenue', alpha=0.9)
ax.bar(x + w/2, df_yoy['TotalMargin'], w, color=TEAL,   label='Gross Margin', alpha=0.9)
ax.set_xticks(x); ax.set_xticklabels(years, rotation=45, ha='right')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
ax.set_title('Revenue vs Gross Margin')
ax.legend(fontsize=8)
ax.set_facecolor(CREAM)

# Chart 2 — Gross Margin %
ax2 = axes[1]
ax2.plot(years, df_yoy['GrossMarginPct'], color=BLUE, marker='o', linewidth=2.5, markersize=6)
ax2.fill_between(range(len(years)), df_yoy['GrossMarginPct'], alpha=0.12, color=BLUE)
for i, v in enumerate(df_yoy['GrossMarginPct']):
    ax2.annotate(f'{v:.1f}%', (i, v), textcoords='offset points', xytext=(0, 8),
                 ha='center', fontsize=8, color=BLUE)
ax2.set_xticks(range(len(years))); ax2.set_xticklabels(years, rotation=45, ha='right')
ax2.set_title('Gross Margin %')
ax2.set_facecolor(CREAM)

# Chart 3 — Orders & Customers
ax3 = axes[2]
ax3.bar(x - w/2, df_yoy['TotalOrders'],     w, color=YELLOW, label='Orders', alpha=0.9)
ax3.bar(x + w/2, df_yoy['UniqueCustomers'], w, color='#C0392B', label='Unique Customers', alpha=0.9)
ax3.set_xticks(x); ax3.set_xticklabels(years, rotation=45, ha='right')
ax3.set_title('Orders & Unique Customers')
ax3.legend(fontsize=8)
ax3.set_facecolor(CREAM)

plt.tight_layout()
add_pomerol_footer(fig)
plt.show()

## 2. Top 10 Products — Revenue & Margin

In [ ]:
df_products = spark.sql("""
    SELECT
        p.ProductName,
        p.CategoryName,
        p.MarginBand,
        p.PriceTier,
        ROUND(SUM(fi.SalesAmount), 2)          AS TotalSales,
        ROUND(SUM(fi.GrossMargin), 2)          AS TotalMargin,
        ROUND(
            SUM(fi.GrossMargin) / NULLIF(SUM(fi.SalesAmount), 0) * 100, 1
        )                                      AS MarginPct,
        SUM(fi.Quantity)                       AS TotalQty
    FROM gold_fact_sales_items fi
    JOIN gold_dim_product p ON fi.ProductID = p.ProductID
    GROUP BY p.ProductName, p.CategoryName, p.MarginBand, p.PriceTier
    ORDER BY TotalSales DESC
    LIMIT 10
""").toPandas()

print(df_products[['ProductName','CategoryName','TotalSales','TotalMargin','MarginPct']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=CREAM)
fig.suptitle('Top 10 Products by Revenue', fontsize=14, fontweight='bold', color=DARK, y=1.01)

df_p = df_products.sort_values('TotalSales')
short_names = df_p['ProductName'].str[:30]

# Chart 1 — Revenue
ax = axes[0]
bars = ax.barh(short_names, df_p['TotalSales'], color=ORANGE, alpha=0.85)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
for bar, val in zip(bars, df_p['TotalSales']):
    ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
            fmt_millions(val, None), va='center', fontsize=8, color=DARK)
ax.set_title('Total Revenue')
ax.set_facecolor(CREAM)

# Chart 2 — Margin %
ax2 = axes[1]
colors = [TEAL if m >= 20 else YELLOW if m >= 15 else '#C0392B' for m in df_p['MarginPct']]
bars2 = ax2.barh(short_names, df_p['MarginPct'], color=colors, alpha=0.85)
ax2.axvline(x=20, color=TEAL,   linestyle='--', linewidth=1, alpha=0.6, label='High (≥20%)')
ax2.axvline(x=15, color=YELLOW, linestyle='--', linewidth=1, alpha=0.6, label='Medium (≥15%)')
for bar, val in zip(bars2, df_p['MarginPct']):
    ax2.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'{val:.1f}%', va='center', fontsize=8, color=DARK)
ax2.set_title('Gross Margin %')
ax2.legend(fontsize=8, loc='lower right')
ax2.set_facecolor(CREAM)

plt.tight_layout()
add_pomerol_footer(fig, 'Aggregated margin uses SUM(GrossMargin)/SUM(SalesAmount)')
plt.show()

## 3. Sales by Division & Region

In [ ]:
df_div = spark.sql("""
    SELECT
        c.DivisionName,
        c.Continent,
        ROUND(SUM(fi.SalesAmount), 2)        AS TotalSales,
        ROUND(SUM(fi.GrossMargin), 2)        AS TotalMargin,
        ROUND(
            SUM(fi.GrossMargin) / NULLIF(SUM(fi.SalesAmount), 0) * 100, 1
        )                                    AS MarginPct,
        COUNT(DISTINCT fi.CustomerID)        AS Customers,
        COUNT(DISTINCT fo.OrderID)           AS Orders
    FROM gold_fact_sales_items fi
    JOIN gold_fact_orders fo   ON fi.OrderID   = fo.OrderID
    JOIN gold_dim_customer c   ON fi.CustomerID = c.CustomerID
    WHERE c.DivisionName IS NOT NULL
    GROUP BY c.DivisionName, c.Continent
    ORDER BY TotalSales DESC
""").toPandas()

print(df_div.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=CREAM)
fig.suptitle('Sales Performance by Division', fontsize=14, fontweight='bold', color=DARK, y=1.01)

# Chart 1 — Revenue by Division
ax = axes[0]
div_colors = sns.color_palette(PALETTE[:len(df_div)], len(df_div))
bars = ax.barh(df_div['DivisionName'], df_div['TotalSales'], color=div_colors, alpha=0.9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
for bar, val in zip(bars, df_div['TotalSales']):
    ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
            fmt_millions(val, None), va='center', fontsize=8, color=DARK)
ax.set_title('Total Revenue by Division')
ax.set_facecolor(CREAM)

# Chart 2 — Revenue share (pie)
ax2 = axes[1]
wedges, texts, autotexts = ax2.pie(
    df_div['TotalSales'],
    labels=df_div['DivisionName'],
    colors=PALETTE[:len(df_div)],
    autopct='%1.1f%%',
    startangle=90,
    pctdistance=0.75,
    wedgeprops={'edgecolor': CREAM, 'linewidth': 2}
)
for t in autotexts:
    t.set_fontsize(8)
ax2.set_title('Revenue Share by Division')
ax2.set_facecolor(CREAM)

plt.tight_layout()
add_pomerol_footer(fig)
plt.show()

## 4. Budget vs Actual by Sales Rep

In [ ]:
df_budget = spark.sql("""
    SELECT
        e.FullName                          AS SalesRep,
        e.Title,
        COALESCE(b.BudgetAmount, 0)         AS Budget,
        ROUND(COALESCE(SUM(fi.SalesAmount), 0), 2) AS ActualSales,
        ROUND(
            COALESCE(SUM(fi.SalesAmount), 0) / NULLIF(b.BudgetAmount, 0) * 100, 1
        )                                   AS AttainmentPct
    FROM gold_dim_employee e
    LEFT JOIN gold_fact_budget b
        ON e.EmployeeID = b.EmployeeID AND b.BudgetYear = 2020
    LEFT JOIN gold_fact_sales_items fi
        ON e.EmployeeID = fi.EmployeeID AND fi.FiscalYear = 2020
    WHERE e.EmployeeID > 0
      AND b.BudgetAmount > 0
    GROUP BY e.FullName, e.Title, b.BudgetAmount
    ORDER BY ActualSales DESC
    LIMIT 12
""").toPandas()

print(df_budget[['SalesRep','Budget','ActualSales','AttainmentPct']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5), facecolor=CREAM)
fig.suptitle('Budget vs Actual Sales — FY2020 by Sales Rep', fontsize=14, fontweight='bold', color=DARK)

df_b = df_budget.sort_values('ActualSales', ascending=False)
x = np.arange(len(df_b))
w = 0.38

bars1 = ax.bar(x - w/2, df_b['Budget'],      w, color=GRAY,   alpha=0.6, label='Budget')
bars2 = ax.bar(x + w/2, df_b['ActualSales'], w, color=ORANGE, alpha=0.9, label='Actual Sales')

# Attainment % annotations
for i, (b, a, pct) in enumerate(zip(df_b['Budget'], df_b['ActualSales'], df_b['AttainmentPct'])):
    color = TEAL if pct >= 100 else '#C0392B' if pct < 80 else YELLOW
    ax.annotate(f'{pct:.0f}%',
                xy=(i + w/2, a),
                xytext=(0, 5), textcoords='offset points',
                ha='center', fontsize=7.5, color=color, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(df_b['SalesRep'], rotation=40, ha='right', fontsize=8)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
ax.set_title('Attainment % shown above Actual bars  |  Green ≥100%  |  Red <80%', fontsize=9, color=GRAY)
ax.legend(fontsize=9)
ax.set_facecolor(CREAM)

plt.tight_layout()
add_pomerol_footer(fig, 'FY2020 only (Budget data scope). BudgetMonthly available in gold_fact_budget.')
plt.show()

## 5. Gross Margin Distribution by Price Tier

In [ ]:
df_tier = spark.sql("""
    SELECT
        p.PriceTier,
        p.MarginBand,
        fi.GrossMarginPct
    FROM gold_fact_sales_items fi
    JOIN gold_dim_product p ON fi.ProductID = p.ProductID
    WHERE fi.GrossMarginPct IS NOT NULL
      AND fi.SalesAmount > 0
""").toPandas()

tier_order = ['Budget', 'Mid-Range', 'Premium', 'Luxury']
df_tier['PriceTier'] = pd.Categorical(df_tier['PriceTier'], categories=tier_order, ordered=True)
print(df_tier.groupby('PriceTier')['GrossMarginPct'].describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=CREAM)
fig.suptitle('Gross Margin Distribution by Price Tier', fontsize=14, fontweight='bold', color=DARK, y=1.01)

tier_colors = {t: c for t, c in zip(tier_order, [TEAL, ORANGE, BLUE, YELLOW])}
palette = [tier_colors[t] for t in tier_order if t in df_tier['PriceTier'].unique()]

# Box plot
ax = axes[0]
sns.boxplot(data=df_tier, x='PriceTier', y='GrossMarginPct',
            order=tier_order, palette=palette, ax=ax, linewidth=1.2)
ax.set_title('Box Plot — Margin % by Price Tier')
ax.set_xlabel('Price Tier')
ax.set_ylabel('Gross Margin %')
ax.axhline(y=0, color='#C0392B', linestyle='--', linewidth=0.8, alpha=0.7)
ax.set_facecolor(CREAM)

# Violin plot
ax2 = axes[1]
sns.violinplot(data=df_tier, x='PriceTier', y='GrossMarginPct',
               order=tier_order, palette=palette, ax=ax2, inner='quartile', linewidth=1)
ax2.set_title('Distribution Shape — Margin % by Price Tier')
ax2.set_xlabel('Price Tier')
ax2.set_ylabel('Gross Margin %')
ax2.axhline(y=0, color='#C0392B', linestyle='--', linewidth=0.8, alpha=0.7)
ax2.set_facecolor(CREAM)

plt.tight_layout()
add_pomerol_footer(fig, 'NOTE: GrossMargin uses current UnitCost. Historical cost changes not tracked (no SCD Type 2 yet).')
plt.show()

## 6. Top 10 Customers by Revenue

In [ ]:
df_cust = spark.sql("""
    SELECT
        c.CustomerName,
        c.Country,
        c.DivisionName,
        COUNT(DISTINCT fo.OrderID)               AS TotalOrders,
        ROUND(SUM(fi.SalesAmount), 2)            AS TotalSales,
        ROUND(SUM(fi.GrossMargin), 2)            AS TotalMargin,
        ROUND(SUM(fi.SalesAmount) / COUNT(DISTINCT fo.OrderID), 2) AS AvgOrderValue
    FROM gold_fact_sales_items fi
    JOIN gold_fact_orders fo   ON fi.OrderID    = fo.OrderID
    JOIN gold_dim_customer c   ON fi.CustomerID = c.CustomerID
    WHERE fi.CustomerID IS NOT NULL
    GROUP BY c.CustomerName, c.Country, c.DivisionName
    ORDER BY TotalSales DESC
    LIMIT 10
""").toPandas()

print(df_cust[['CustomerName','Country','TotalOrders','TotalSales','AvgOrderValue']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=CREAM)
fig.suptitle('Top 10 Customers by Revenue', fontsize=14, fontweight='bold', color=DARK, y=1.01)

df_c = df_cust.sort_values('TotalSales')
short = df_c['CustomerName'].str[:28]

# Revenue bar
ax = axes[0]
div_colors = [PALETTE[['North America','Europe','South America','Central America',
                        'Nordic','Asia Pacific','Oceania','Unknown']
                       .index(d) % len(PALETTE)] if d in
              ['North America','Europe','South America','Central America',
               'Nordic','Asia Pacific','Oceania','Unknown']
              else ORANGE for d in df_c['DivisionName']]
bars = ax.barh(short, df_c['TotalSales'], color=ORANGE, alpha=0.85)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
for bar, val in zip(bars, df_c['TotalSales']):
    ax.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
            fmt_millions(val, None), va='center', fontsize=8, color=DARK)
ax.set_title('Total Revenue')
ax.set_facecolor(CREAM)

# Avg Order Value scatter
ax2 = axes[1]
scatter = ax2.scatter(df_c['TotalSales'], df_c['AvgOrderValue'],
                      s=df_c['TotalOrders'] * 5, color=BLUE, alpha=0.75, edgecolors=DARK, linewidth=0.5)
for _, row in df_c.iterrows():
    ax2.annotate(row['CustomerName'][:18],
                 (row['TotalSales'], row['AvgOrderValue']),
                 textcoords='offset points', xytext=(5, 3),
                 fontsize=7, color=DARK)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
ax2.set_xlabel('Total Revenue')
ax2.set_ylabel('Avg Order Value')
ax2.set_title('Revenue vs Avg Order Value  (bubble = order count)')
ax2.set_facecolor(CREAM)

plt.tight_layout()
add_pomerol_footer(fig)
plt.show()

## 7. Seasonal Sales Pattern — Month × Year Heatmap

In [ ]:
df_heat = spark.sql("""
    SELECT
        fi.OrderYear,
        fi.OrderMonth,
        d.MonthName,
        ROUND(SUM(fi.SalesAmount), 2) AS TotalSales,
        COUNT(DISTINCT fo.OrderID)    AS TotalOrders
    FROM gold_fact_sales_items fi
    JOIN gold_fact_orders fo  ON fi.OrderID  = fo.OrderID
    JOIN gold_dim_date d      ON fi.OrderDateKey = d.DateKey
    GROUP BY fi.OrderYear, fi.OrderMonth, d.MonthName
    ORDER BY fi.OrderYear, fi.OrderMonth
""").toPandas()

# Pivot for heatmap
month_order = ['January','February','March','April','May','June',
               'July','August','September','October','November','December']
pivot = df_heat.pivot_table(index='MonthName', columns='OrderYear', values='TotalSales', aggfunc='sum')
pivot = pivot.reindex([m for m in month_order if m in pivot.index])
print(pivot.round(0))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor=CREAM)
fig.suptitle('Seasonal Sales Pattern — ACME Inc.', fontsize=14, fontweight='bold', color=DARK, y=1.01)

# Heatmap
ax = axes[0]
sns.heatmap(pivot / 1000, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, linecolor=CREAM, ax=ax,
            cbar_kws={'label': 'Revenue ($K)'})
ax.set_title('Monthly Revenue ($K) by Year')
ax.set_ylabel('')
ax.set_xlabel('Year')
ax.set_facecolor(CREAM)

# Monthly average line
ax2 = axes[1]
monthly_avg = df_heat.groupby(['OrderMonth','MonthName'])['TotalSales'].mean().reset_index()
monthly_avg = monthly_avg.sort_values('OrderMonth')
ax2.plot(monthly_avg['MonthName'], monthly_avg['TotalSales'],
         color=ORANGE, marker='o', linewidth=2.5, markersize=7)
ax2.fill_between(range(len(monthly_avg)), monthly_avg['TotalSales'],
                 alpha=0.15, color=ORANGE)
ax2.set_xticks(range(len(monthly_avg)))
ax2.set_xticklabels([m[:3] for m in monthly_avg['MonthName']], rotation=45, ha='right')
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
ax2.set_title('Average Monthly Revenue (all years)')
ax2.set_facecolor(CREAM)

# Highlight peak month
peak_idx = monthly_avg['TotalSales'].idxmax()
peak_val = monthly_avg.loc[peak_idx, 'TotalSales']
peak_name = monthly_avg.loc[peak_idx, 'MonthName']
ax2.annotate(f'Peak: {peak_name}',
             xy=(monthly_avg.index.get_loc(peak_idx), peak_val),
             xytext=(10, -20), textcoords='offset points',
             fontsize=9, color=ORANGE, fontweight='bold',
             arrowprops={'arrowstyle': '->', 'color': ORANGE})

plt.tight_layout()
add_pomerol_footer(fig, 'Insight: Seasonal peaks inform inventory planning opportunity')
plt.show()

## 8. Data Quality Summary

In [ ]:
df_dq = spark.sql("""
    SELECT
        severity,
        check_name,
        table_name,
        COUNT(*) AS issue_count
    FROM gold_dq_issues
    GROUP BY severity, check_name, table_name
    ORDER BY
        CASE severity
            WHEN 'Critical' THEN 1
            WHEN 'High'     THEN 2
            WHEN 'Medium'   THEN 3
            ELSE 4
        END,
        issue_count DESC
""").toPandas()

df_dq_sev = df_dq.groupby('severity')['issue_count'].sum().reset_index()
print(df_dq_sev)
print()
print(df_dq.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=CREAM)
fig.suptitle('Data Quality Summary — ACME Inc.', fontsize=14, fontweight='bold', color=DARK, y=1.01)

sev_order  = ['Critical', 'High', 'Medium', 'Info']
sev_colors = {'Critical': '#C0392B', 'High': ORANGE, 'Medium': YELLOW, 'Info': TEAL}

df_sev = df_dq_sev.set_index('severity').reindex(sev_order).dropna().reset_index()
colors = [sev_colors.get(s, GRAY) for s in df_sev['severity']]

# Donut chart — severity breakdown
ax = axes[0]
wedges, texts, autotexts = ax.pie(
    df_sev['issue_count'],
    labels=df_sev['severity'],
    colors=colors,
    autopct='%1.0f%%',
    startangle=90,
    pctdistance=0.75,
    wedgeprops={'edgecolor': CREAM, 'linewidth': 3, 'width': 0.55}  # donut
)
total = df_sev['issue_count'].sum()
ax.text(0, 0, f'{total:,}\nissues', ha='center', va='center',
        fontsize=11, fontweight='bold', color=DARK)
for t in autotexts:
    t.set_fontsize(8)
ax.set_title('Issues by Severity')
ax.set_facecolor(CREAM)

# Bar chart — by check name
ax2 = axes[1]
df_check = df_dq.sort_values('issue_count', ascending=True)
bar_colors = [sev_colors.get(s, GRAY) for s in df_check['severity']]
bars = ax2.barh(df_check['check_name'], df_check['issue_count'], color=bar_colors, alpha=0.85)
for bar, val in zip(bars, df_check['issue_count']):
    ax2.text(bar.get_width() + max(df_check['issue_count']) * 0.01,
             bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=8, color=DARK)
ax2.set_title('Issue Count by Check Type')
ax2.set_xlabel('Number of Records Affected')
ax2.set_facecolor(CREAM)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=s) for s, c in sev_colors.items()]
ax2.legend(handles=legend_elements, fontsize=8, loc='lower right')

plt.tight_layout()
add_pomerol_footer(fig, 'Excludes shipment_before_order and unitprice_discrepancy (known/expected)')
plt.show()

## 9. Drill-Down: Product Line → Product → Customer

**Mapping:** *Product Line* = `CategoryName` (no separate ProductLine field in source).

Hierarchy: `CategoryName` → `ProductName` → `CustomerName`

Fully supported via `gold_dim_product` joined to `gold_fact_sales_items` and `gold_dim_customer`.

In [ ]:
df_drill = spark.sql("""
    SELECT
        p.CategoryName AS ProductLine,
        p.ProductName,
        c.CustomerName,
        c.Country,
        c.DivisionName,
        SUM(fi.Quantity)                            AS TotalQty,
        ROUND(SUM(fi.SalesAmount), 2)              AS TotalSales,
        ROUND(SUM(fi.GrossMargin), 2)              AS TotalMargin
    FROM gold_fact_sales_items fi
    JOIN gold_dim_product  p ON fi.ProductID  = p.ProductID
    JOIN gold_dim_customer c ON fi.CustomerID = c.CustomerID
    GROUP BY p.CategoryName, p.ProductName, c.CustomerName, c.Country, c.DivisionName
    ORDER BY TotalSales DESC
""").toPandas()

df_line = df_drill.groupby('ProductLine').agg(
    TotalSales=('TotalSales','sum'),
    TotalQty=('TotalQty','sum'),
    Products=('ProductName','nunique'),
    Customers=('CustomerName','nunique')
).reset_index().sort_values('TotalSales', ascending=False)

df_prod_grp = df_drill.groupby(['ProductLine','ProductName']).agg(
    TotalSales=('TotalSales','sum'), TotalQty=('TotalQty','sum')
).reset_index().sort_values('TotalSales', ascending=False)

print('=== Level 1: Product Line (Category) ===')
print(df_line.to_string(index=False))
print('\n=== Level 2: Top Products per Line ===')
print(df_prod_grp.head(20).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), facecolor=CREAM)
fig.suptitle('Drill-Down: Product Line → Product (Qty vs Revenue)', fontsize=14, fontweight='bold', color=DARK, y=1.01)

ax = axes[0]
df_l = df_line.sort_values('TotalSales')
clrs = [PALETTE[i % len(PALETTE)] for i in range(len(df_l))]
bars = ax.barh(df_l['ProductLine'], df_l['TotalSales'], color=clrs, alpha=0.88)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
for bar, qty, val in zip(bars, df_l['TotalQty'], df_l['TotalSales']):
    ax.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2,
            f'{fmt_millions(val,None)}  ({qty:,} units)', va='center', fontsize=7.5, color=DARK)
ax.set_title('Revenue & Qty by Product Line')
ax.set_facecolor(CREAM)

ax2 = axes[1]
top_prods = df_prod_grp.head(30)
lines_uniq = top_prods['ProductLine'].unique()
lc = {line: PALETTE[i % len(PALETTE)] for i, line in enumerate(lines_uniq)}
for line, grp in top_prods.groupby('ProductLine'):
    ax2.scatter(grp['TotalQty'], grp['TotalSales'], label=line,
                color=lc[line], s=60, alpha=0.8, edgecolors=DARK, linewidth=0.4)
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,p: f'{x:,.0f}'))
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
ax2.set_xlabel('Total Quantity Sold')
ax2.set_ylabel('Total Revenue')
ax2.set_title('Qty vs Revenue by Product (Top 30)')
ax2.legend(fontsize=7, loc='upper left')
ax2.set_facecolor(CREAM)

plt.tight_layout()
add_pomerol_footer(fig, 'Product Line = CategoryName in this dataset')
plt.show()

## 10. Quantity Sold vs Sales — All Business Dimensions

Covering: Country, State/Province, Supplier, Sales Rep

In [ ]:
df_geo = spark.sql("""
    SELECT c.Country, c.StateProvince, c.DivisionName,
           SUM(fi.Quantity) AS TotalQty,
           ROUND(SUM(fi.SalesAmount),2) AS TotalSales,
           COUNT(DISTINCT fi.CustomerID) AS Customers
    FROM gold_fact_sales_items fi
    JOIN gold_dim_customer c ON fi.CustomerID = c.CustomerID
    GROUP BY c.Country, c.StateProvince, c.DivisionName
    ORDER BY TotalSales DESC
""").toPandas()

df_supplier = spark.sql("""
    SELECT p.SupplierName, p.SupplierCountry, p.SupplierContinent,
           SUM(fi.Quantity) AS TotalQty,
           ROUND(SUM(fi.SalesAmount),2) AS TotalSales,
           COUNT(DISTINCT p.ProductID) AS Products
    FROM gold_fact_sales_items fi
    JOIN gold_dim_product p ON fi.ProductID = p.ProductID
    GROUP BY p.SupplierName, p.SupplierCountry, p.SupplierContinent
    ORDER BY TotalSales DESC LIMIT 15
""").toPandas()

df_rep = spark.sql("""
    SELECT e.FullName AS SalesRep, e.Title, e.OfficeCountry,
           SUM(fi.Quantity) AS TotalQty,
           ROUND(SUM(fi.SalesAmount),2) AS TotalSales,
           COUNT(DISTINCT fo.OrderID) AS Orders
    FROM gold_fact_sales_items fi
    JOIN gold_fact_orders fo  ON fi.OrderID    = fo.OrderID
    JOIN gold_dim_employee e  ON fi.EmployeeID = e.EmployeeID
    WHERE fi.EmployeeID > 0
    GROUP BY e.FullName, e.Title, e.OfficeCountry
    ORDER BY TotalSales DESC
""").toPandas()

print('Top 10 Countries:'); print(df_geo.head(10)[['Country','StateProvince','TotalQty','TotalSales']].to_string(index=False))
print('\nSuppliers:');       print(df_supplier[['SupplierName','SupplierCountry','TotalQty','TotalSales']].to_string(index=False))
print('\nSales Reps:');      print(df_rep[['SalesRep','TotalQty','TotalSales','Orders']].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor=CREAM)
fig.suptitle('Quantity Sold vs Revenue — All Business Dimensions', fontsize=14, fontweight='bold', color=DARK)

# Top 15 Countries
ax = axes[0][0]
df_g = df_geo.head(15).sort_values('TotalSales')
ax.barh(df_g['Country'], df_g['TotalSales'], color=ORANGE, alpha=0.8)
ax_t = ax.twiny()
ax_t.plot(df_g['TotalQty'], range(len(df_g)), 'o--', color=BLUE, linewidth=1.5, markersize=5)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
ax.set_title('Top 15 Countries — Revenue (bars) vs Qty (dots)')
ax.set_facecolor(CREAM)

# USA State/Province
ax2 = axes[0][1]
df_us = df_geo[df_geo['Country']=='USA'].sort_values('TotalSales', ascending=False).head(10)
if len(df_us) > 0:
    df_us_s = df_us.sort_values('TotalSales')
    ax2.barh(df_us_s['StateProvince'], df_us_s['TotalSales'], color=BLUE, alpha=0.8)
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
    ax2.set_title('USA — Revenue by State/Province')
else:
    ax2.text(0.5, 0.5, 'No US State/Province data\n(check StateProvince field)', ha='center', va='center',
             transform=ax2.transAxes, color=GRAY, fontsize=11)
    ax2.set_title('USA — Revenue by State/Province')
ax2.set_facecolor(CREAM)

# Supplier
ax3 = axes[1][0]
df_s = df_supplier.sort_values('TotalSales')
ax3.barh(df_s['SupplierName'].str[:22], df_s['TotalSales'], color=TEAL, alpha=0.85)
ax3_t = ax3.twiny()
ax3_t.plot(df_s['TotalQty'], range(len(df_s)), 's--', color=ORANGE, markersize=5, linewidth=1.2)
ax3.xaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
ax3.set_title('Supplier — Revenue (bars) vs Qty (dots)')
ax3.set_facecolor(CREAM)

# Sales Rep
ax4 = axes[1][1]
df_r = df_rep.sort_values('TotalSales', ascending=False).head(12)
x_r  = np.arange(len(df_r))
ax4.bar(x_r, df_r['TotalSales'], color=ORANGE, alpha=0.85)
ax4_t = ax4.twinx()
ax4_t.plot(x_r, df_r['TotalQty'], 'o--', color=BLUE, linewidth=2, markersize=6)
ax4.set_xticks(x_r)
ax4.set_xticklabels(df_r['SalesRep'].str.split().str[-1], rotation=40, ha='right', fontsize=8)
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_millions))
ax4.set_title('Sales Rep — Revenue (bars) vs Qty (line)')
ax4.set_facecolor(CREAM)

plt.tight_layout()
add_pomerol_footer(fig)
plt.show()

## 11. [BONUS] Customers YTD vs LY YTD

Reference date: April 1, 2021. YTD = Jan 1–Mar 30, 2021. LY YTD = Jan 1–Mar 30, 2020.

In [ ]:
df_ytd = spark.sql("""
    SELECT period,
           COUNT(DISTINCT CustomerID) AS UniqueCustomers,
           COUNT(DISTINCT OrderID)   AS TotalOrders,
           ROUND(SUM(SalesAmount),2) AS TotalSales
    FROM (
        SELECT fi.CustomerID, fo.OrderID, fi.SalesAmount,
               CASE
                   WHEN fi.OrderYear=2021 AND fi.OrderMonth BETWEEN 1 AND 3 THEN 'YTD 2021 (Jan-Mar)'
                   WHEN fi.OrderYear=2020 AND fi.OrderMonth BETWEEN 1 AND 3 THEN 'LY YTD 2020 (Jan-Mar)'
               END AS period
        FROM gold_fact_sales_items fi
        JOIN gold_fact_orders fo ON fi.OrderID = fo.OrderID
        WHERE (fi.OrderYear=2021 AND fi.OrderMonth BETWEEN 1 AND 3)
           OR (fi.OrderYear=2020 AND fi.OrderMonth BETWEEN 1 AND 3)
    ) t WHERE period IS NOT NULL
    GROUP BY period ORDER BY period DESC
""").toPandas()

df_ytd_m = spark.sql("""
    SELECT fi.OrderYear, fi.OrderMonth, d.MonthName,
           COUNT(DISTINCT fi.CustomerID) AS UniqueCustomers,
           COUNT(DISTINCT fo.OrderID)   AS Orders,
           ROUND(SUM(fi.SalesAmount),2) AS Sales
    FROM gold_fact_sales_items fi
    JOIN gold_fact_orders fo ON fi.OrderID      = fo.OrderID
    JOIN gold_dim_date d     ON fi.OrderDateKey = d.DateKey
    WHERE (fi.OrderYear=2021 AND fi.OrderMonth BETWEEN 1 AND 3)
       OR (fi.OrderYear=2020 AND fi.OrderMonth BETWEEN 1 AND 3)
    GROUP BY fi.OrderYear, fi.OrderMonth, d.MonthName
    ORDER BY fi.OrderYear, fi.OrderMonth
""").toPandas()

print('=== YTD vs LY YTD Summary ===')
print(df_ytd.to_string(index=False))
print('\n=== Monthly Detail ===')
print(df_ytd_m.to_string(index=False))
if len(df_ytd) == 2:
    ytd  = df_ytd[df_ytd['period'].str.contains('2021')].iloc[0]
    lytd = df_ytd[df_ytd['period'].str.contains('2020')].iloc[0]
    print(f'\nCustomer YoY: {ytd.UniqueCustomers} vs {lytd.UniqueCustomers} ({(ytd.UniqueCustomers/lytd.UniqueCustomers-1)*100:+.1f}%)')
    print(f'Sales YoY:    {fmt_millions(ytd.TotalSales,None)} vs {fmt_millions(lytd.TotalSales,None)} ({(ytd.TotalSales/lytd.TotalSales-1)*100:+.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=CREAM)
fig.suptitle('YTD vs LY YTD — Customers & Revenue (Jan-Mar)', fontsize=14, fontweight='bold', color=DARK, y=1.01)

ax = axes[0]
if len(df_ytd) == 2:
    metrics = ['UniqueCustomers','TotalOrders']
    labels  = ['Unique Customers','Total Orders']
    x_pos   = np.arange(len(labels))
    w = 0.35
    ytd_v  = [df_ytd[df_ytd['period'].str.contains('2021')][m].values[0] for m in metrics]
    lytd_v = [df_ytd[df_ytd['period'].str.contains('2020')][m].values[0] for m in metrics]
    ax.bar(x_pos-w/2, lytd_v, w, color=GRAY,   alpha=0.7, label='LY YTD 2020')
    ax.bar(x_pos+w/2, ytd_v,  w, color=ORANGE, alpha=0.9, label='YTD 2021')
    for i, (ly, yv) in enumerate(zip(lytd_v, ytd_v)):
        delta = (yv/ly-1)*100 if ly else 0
        ax.annotate(f'{delta:+.1f}%', xy=(i+w/2, yv), xytext=(0,6),
                    textcoords='offset points', ha='center', fontsize=10,
                    color=TEAL if delta>=0 else '#C0392B', fontweight='bold')
    ax.set_xticks(x_pos); ax.set_xticklabels(labels)
    ax.set_title('YTD vs LY YTD Comparison')
    ax.legend(fontsize=9)
ax.set_facecolor(CREAM)

ax2 = axes[1]
for yr, color, lbl in [(2021, ORANGE, '2021 YTD'), (2020, GRAY, '2020 LY YTD')]:
    sub = df_ytd_m[df_ytd_m['OrderYear']==yr].sort_values('OrderMonth')
    if len(sub) > 0:
        ax2.plot(sub['MonthName'], sub['UniqueCustomers'],
                 color=color, marker='o', linewidth=2.5, markersize=7, label=lbl)
        for _, row in sub.iterrows():
            ax2.annotate(str(int(row['UniqueCustomers'])), (row['MonthName'], row['UniqueCustomers']),
                         xytext=(0,7), textcoords='offset points', ha='center', fontsize=8, color=color)
ax2.set_title('Unique Customers per Month — 2021 vs 2020')
ax2.set_ylabel('Unique Customers')
ax2.legend(fontsize=9)
ax2.set_facecolor(CREAM)

plt.tight_layout()
add_pomerol_footer(fig)
plt.show()

## 12. [BONUS] Delivery Performance (Order Date vs Shipment Date)

> **WARNING:** ShipmentDates = 2007-2012; OrderDates = 2016-2021.
> All DeliveryDays are **negative** — shipment data from a legacy system.
> Real SLA analysis is **not possible**. Issue documented for the client.

In [ ]:
df_del = spark.sql("""
    SELECT fo.OrderDate, fo.ShipmentDate, fo.DeliveryDays,
           sh.ShipperName, fo.OrderYear
    FROM gold_fact_orders fo
    LEFT JOIN gold_dim_shipper sh ON fo.ShipperID = sh.ShipperID
    WHERE fo.ShipmentDate IS NOT NULL
""").toPandas()

df_sh = spark.sql("""
    SELECT sh.ShipperName, COUNT(fo.OrderID) AS Orders,
           ROUND(AVG(fo.DeliveryDays),1) AS AvgDays,
           MIN(fo.DeliveryDays) AS MinDays, MAX(fo.DeliveryDays) AS MaxDays
    FROM gold_fact_orders fo
    LEFT JOIN gold_dim_shipper sh ON fo.ShipperID = sh.ShipperID
    WHERE fo.DeliveryDays IS NOT NULL
    GROUP BY sh.ShipperName ORDER BY Orders DESC
""").toPandas()

print(df_del['DeliveryDays'].describe().round(1))
print(f'All negative: {(df_del["DeliveryDays"] < 0).all()}')
print(f'OrderDate:    {df_del["OrderDate"].min()} to {df_del["OrderDate"].max()}')
print(f'ShipmentDate: {df_del["ShipmentDate"].min()} to {df_del["ShipmentDate"].max()}')
print('\nBy Shipper:')
print(df_sh.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=CREAM)
fig.suptitle('Delivery Performance — LEGACY DATA ISSUE', fontsize=14, fontweight='bold', color='#C0392B', y=1.01)

ax = axes[0]
ax.hist(df_del['DeliveryDays'], bins=40, color='#C0392B', alpha=0.7, edgecolor=CREAM)
ax.axvline(x=0, color=DARK, linestyle='--', linewidth=1.5, label='Order Date (zero ref)')
ax.axvline(x=df_del['DeliveryDays'].mean(), color=ORANGE, linestyle='-', linewidth=2,
           label=f'Mean: {df_del["DeliveryDays"].mean():.0f} days')
ax.set_xlabel('Delivery Days (negative = shipment predates order)')
ax.set_ylabel('Number of Orders')
ax.set_title('All DeliveryDays Negative\n(Shipments 2007-2012 vs Orders 2016-2021)')
ax.legend(fontsize=8)
ax.set_facecolor(CREAM)

ax2 = axes[1]
clrs2 = [ORANGE, TEAL, BLUE, GRAY, GRAY][:len(df_sh)]
ax2.bar(df_sh['ShipperName'], df_sh['Orders'], color=clrs2, alpha=0.85)
for i, (_, row) in enumerate(df_sh.iterrows()):
    ax2.text(i, row['Orders']+30, f'{row["Orders"]:,}\norders', ha='center', fontsize=8, color=DARK)
ax2.set_title('Orders by Shipper\n(Volume only — SLA cannot be calculated)')
ax2.set_ylabel('Number of Orders')
ax2.set_xticklabels(df_sh['ShipperName'], rotation=20, ha='right')
ax2.set_facecolor(CREAM)

plt.tight_layout()
add_pomerol_footer(fig, 'ACTION: Replace Shipments with current system data to enable real SLA analysis')
plt.show()

## 13. [BONUS] Why Does UnitPrice Appear in Two Tables?

| Table | Column | Meaning |
|---|---|---|
| `silver_order_details` | `UnitPrice` | **Historical price** at time of sale |
| `silver_products` | `UnitPrice` | **Current catalog price** |

This is the **SCD (Slowly Changing Dimension)** pattern. 99.4% of rows differ — expected, not a bug.

**Production fix:** SCD Type 2 on Products (`Valid_From`, `Valid_To`, `Is_Current`).

**Other duplicate data:** `Shipments` contains `CustomerID`, `ProductID`, `EmployeeID`
already in `Orders`/`Order_Details`. Cross-validated in NB_04 — results in `gold_dq_issues`.

In [ ]:
df_price = spark.sql("""
    SELECT p.ProductName,
           cat.CategoryName,
           ROUND(p.UnitPrice, 2)                 AS CurrentCatalogPrice,
           ROUND(AVG(od.UnitPrice), 2)           AS AvgHistoricalPrice,
           ROUND(MIN(od.UnitPrice), 2)           AS MinSalePrice,
           ROUND(MAX(od.UnitPrice), 2)           AS MaxSalePrice,
           COUNT(od.OrderID)                     AS Transactions,
           SUM(CASE WHEN od.UnitPrice <> p.UnitPrice THEN 1 ELSE 0 END) AS Mismatches,
           ROUND(
               SUM(CASE WHEN od.UnitPrice <> p.UnitPrice THEN 1 ELSE 0 END)
               * 100.0 / COUNT(od.OrderID), 1
           )                                     AS MismatchPct
    FROM silver_order_details od
    JOIN silver_products   p   ON od.ProductID  = p.ProductID
    JOIN silver_categories cat ON p.CategoryID  = cat.CategoryID
    GROUP BY p.ProductName, cat.CategoryName, p.UnitPrice
    ORDER BY MismatchPct DESC
    LIMIT 20
""").toPandas()

df_dup = spark.sql("""
    SELECT check_name, severity, COUNT(*) AS issue_count
    FROM gold_dq_issues
    WHERE check_name IN ('shipment_customer_mismatch','shipment_product_mismatch',
                         'shipment_employee_mismatch','unitprice_discrepancy')
    GROUP BY check_name, severity ORDER BY issue_count DESC
""").toPandas()

print('=== UnitPrice: Current vs Historical ===')
print(df_price[['ProductName','CurrentCatalogPrice','AvgHistoricalPrice','Transactions','MismatchPct']].to_string(index=False))
print('\n=== Duplicate Data Cross-Validation ===')
print(df_dup.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), facecolor=CREAM)
fig.suptitle('UnitPrice: Historical vs Current Catalog — SCD Pattern', fontsize=14, fontweight='bold', color=DARK, y=1.01)

ax = axes[0]
ax.scatter(df_price['CurrentCatalogPrice'], df_price['AvgHistoricalPrice'],
           s=df_price['Transactions']*2, color=ORANGE, alpha=0.7, edgecolors=DARK, linewidth=0.4)
mx = max(df_price['CurrentCatalogPrice'].max(), df_price['AvgHistoricalPrice'].max())
ax.plot([0, mx], [0, mx], '--', color=GRAY, linewidth=1.5, alpha=0.7, label='Perfect match line')
ax.set_xlabel('Current Catalog Price ($)')
ax.set_ylabel('Avg Historical Sale Price ($)')
ax.set_title('Current vs Historical Price (bubble = transactions)')
ax.legend(fontsize=8)
ax.set_facecolor(CREAM)

ax2 = axes[1]
df_pm = df_price.sort_values('MismatchPct', ascending=True).tail(15)
clrs_m = [TEAL if v==0 else ORANGE if v<100 else '#C0392B' for v in df_pm['MismatchPct']]
ax2.barh(df_pm['ProductName'].str[:25], df_pm['MismatchPct'], color=clrs_m, alpha=0.85)
ax2.axvline(x=100, color=GRAY, linestyle='--', linewidth=1, alpha=0.7)
ax2.set_xlabel('% Transactions with Price Mismatch')
ax2.set_title('Price Mismatch Rate per Product\n(SCD behavior — not a data error)')
ax2.set_facecolor(CREAM)

plt.tight_layout()
add_pomerol_footer(fig, 'RECOMMENDATION: Implement SCD Type 2 on Products (Valid_From, Valid_To, Is_Current)')
plt.show()

## Requirements Coverage Summary

### Business Dimensions
| Dimension | Gold Column | Section |
|---|---|---|
| Division | `dim_customer.DivisionName` | 3 |
| Country | `dim_customer.Country` | 10 |
| State/Province | `dim_customer.StateProvince` | 10 |
| Sales Rep | `dim_employee.FullName` | 4, 10 |
| Product Line | `dim_product.CategoryName` | 9 |
| Product Category | `dim_product.CategoryName` | 2, 9 |
| Product | `dim_product.ProductName` | 2, 9, 10 |
| Customer | `dim_customer.CustomerName` | 6, 9 |
| Supplier | `dim_product.SupplierName` | 10 |

### Business Questions
| Question | Section |
|---|---|
| Avg sales per transaction | 1 |
| Avg sales per customer | 6 |
| Drill-down Product Group → Product → Customer | 9 |
| Qty vs Sales by Product/Region/Sales Rep/Supplier | 10 |
| [BONUS] YTD vs LY YTD customers | 11 |
| [BONUS] Delivery performance | 12 |
| [BONUS] UnitPrice in two tables (SCD) | 13 |
| [BONUS] Other duplicate data (Shipments) | 13 + NB_04 |

---
*ACME Inc. POC — Pomerol — April 1, 2021*